# 02 — Baselines

* **B1** TF--IDF (word 1-2 grams, class-weighted) + Logistic Regression.
* **B2** TF--IDF (char 3-5 grams) + Logistic Regression — robust to
  misspellings, which are common in user reviews.
* **B3** LightGBM on TF-IDF-SVD + stylometry.

All run in minutes on CPU and establish honest non-neural reference numbers.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(root))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(context="talk", style="whitegrid")
RANDOM_STATE = 0

In [ ]:
tr = pd.read_parquet("../data/train_split.parquet")
va = pd.read_parquet("../data/val_split.parquet")
te = pd.read_parquet("../data/test_full.parquet")
tr.shape, va.shape, te.shape

## B1 — Word TF-IDF + LR

In [ ]:
from src.models import tfidf_lr_pipeline
from src.eval import summarize, dump_metrics, report_text
pipe = tfidf_lr_pipeline()
pipe.fit(tr["text"], tr["label"])
yp_va = pipe.predict(va["text"])
m = summarize("B1_tfidf_word_lr", va["label"], yp_va); print(m); dump_metrics({"B1_tfidf_word_lr": m})
print(report_text(va["label"], yp_va))

## B2 — Char TF-IDF + LR

In [ ]:
from src.models import tfidf_char_lr_pipeline
pipe_c = tfidf_char_lr_pipeline()
pipe_c.fit(tr["text"], tr["label"])
yp_va = pipe_c.predict(va["text"])
m = summarize("B2_tfidf_char_lr", va["label"], yp_va); print(m); dump_metrics({"B2_tfidf_char_lr": m})

## B3 — SVD + stylometry + LightGBM

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from src.features import stylometry_features
import lightgbm as lgb

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.98, sublinear_tf=True, max_features=200_000)
X_tr_bow = vec.fit_transform(tr["text"]); X_va_bow = vec.transform(va["text"])
svd = TruncatedSVD(n_components=200, random_state=0); X_tr_svd = svd.fit_transform(X_tr_bow); X_va_svd = svd.transform(X_va_bow)
S_tr = stylometry_features(tr["text"]); S_va = stylometry_features(va["text"])
X_tr = np.hstack([X_tr_svd, S_tr.values]); X_va = np.hstack([X_va_svd, S_va.values])
model = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.05, num_leaves=63, class_weight="balanced", random_state=0)
model.fit(X_tr, tr["label"], eval_set=[(X_va, va["label"])])
yp_va = model.predict(X_va)
m = summarize("B3_svd_sty_lgbm", va["label"], yp_va); print(m); dump_metrics({"B3_svd_sty_lgbm": m})

## Pick best baseline → submission

In [ ]:
# Produce a submission from the strongest baseline (we'll beat it in later notebooks).
from src.data import write_submission, IDX2LABEL
best = pipe   # swap to pipe_c / model as results dictate
pred_test_label = best.predict(te["text"])
pred_rate = np.array([IDX2LABEL[i] for i in pred_test_label])
out = write_submission(te["index"].values, pred_rate, "submission_B1.csv")
print("wrote", out)